In [ ]:
cd /content/drive/MyDrive/model_exp3

/content/drive/MyDrive/model_exp3


In [ ]:
!ls

 benchmark_vlm.py
 compare_models.py
 config.yaml
 finetuning_scripts
'initial_evalaution_(before_FT_n_KD)_run.ipynb'
 KD_pipeline.ipynb
 knowledge_distillation
 requirements.txt
 results_before_FT_n_KD_100samples
 results_qwen_finetuned_spatial_cust_data_worse_than_before
 results_qwen_finetuned_subset_CV_data_better_than_before
 run_all.py
 run_model.py
 test_script
 unsloth_compiled_cache
 vlm_bench


In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.1
!pip install --no-deps trl==0.22.2

In [ ]:
!python -m knowledge_distillation.generate_teacher_labels \
    --model_path "abhi26/subCV_qwen3-8B_lora" \
    --dataset "nyu-visionx/CV-Bench" \
    --split test \
    --no_rationale\
    --output /content/drive/MyDrive/model_exp3/knowledge_distillation/teacher_outputs/teacher_labels.jsonl \
    --features_dir /content/drive/MyDrive/model_exp3/knowledge_distillation/teacher_outputs/visual_features

  STEP 1: Generate Teacher Labels
  Model    : abhi26/subCV_qwen3-8B_lora
  Dataset  : nyu-visionx/CV-Bench [test] → 2D only
  Output   : /content/drive/MyDrive/model_exp3/knowledge_distillation/teacher_outputs/teacher_labels.jsonl
  Rationale: True
  Logits   : True
Samples: 1438
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-05-27 14:44:28.417585: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
🦥 Unsloth Zoo will now patch everything to make training faster!
[unsloth_zoo.log|WARNING]Device does not support bfloat16. Will change to float16.
==((====))==  Unsloth 2026.5.8: Fast Qwen3_Vl patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 

In [ ]:
cd /content/drive/MyDrive/model_exp3

#Step 2 — Train the student model

In [ ]:

!python -m knowledge_distillation.train_student_internvl \
    --student_model "OpenGVLab/InternVL3_5-1B-Instruct" \
    --teacher_labels /content/drive/MyDrive/model_exp3/knowledge_distillation/teacher_outputs/teacher_labels.jsonl \
    --dataset "nyu-visionx/CV-Bench" \
    --split test \
    --kd_mode response \
    --use_rationale \
    --epochs 3 \
    --batch_size 2 \
    --lr 2e-4

#Step 3 — Evaluate the distilled student

In [ ]:
!python -m knowledge_distillation.evaluate_student \
    --student_path "OpenGVLab/InternVL3_5-1B-Instruct" \
    --adapter_path /content/drive/MyDrive/model_exp3/knowledge_distillation/student_checkpoints/final \
    --dataset "nyu-visionx/CV-Bench" \
    --split test